In [1]:
import os
import glob
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import random
from datetime import datetime, timedelta
from dateutil.relativedelta import relativedelta
import pprint
import pyspark
import pyspark.sql.functions as F
import re

from pyspark.sql.functions import col, udf
from pyspark.sql.types import StringType, IntegerType, FloatType, DateType

import utils.data_processing_bronze_table
import utils.data_processing_silver_table
import utils.data_processing_gold_table

In [23]:
pd.set_option('display.float_format', '{:.2f}'.format)

In [82]:
# Initialize SparkSession
spark = pyspark.sql.SparkSession.builder \
    .appName("dev") \
    .master("local[*]") \
    .getOrCreate()

# Set log level to ERROR to hide warnings
spark.sparkContext.setLogLevel("ERROR")

In [78]:
# set up config
snapshot_date_str = "2023-01-01"

start_date_str = "2023-01-01"
end_date_str = "2025-11-01"

# generate list of dates to process
def generate_first_of_month_dates(start_date_str, end_date_str):
    # Convert the date strings to datetime objects
    start_date = datetime.strptime(start_date_str, "%Y-%m-%d")
    end_date = datetime.strptime(end_date_str, "%Y-%m-%d")
    
    # List to store the first of month dates
    first_of_month_dates = []

    # Start from the first of the month of the start_date
    current_date = datetime(start_date.year, start_date.month, 1)

    while current_date <= end_date:
        # Append the date in yyyy-mm-dd format
        first_of_month_dates.append(current_date.strftime("%Y-%m-%d"))
        
        # Move to the first of the next month
        if current_date.month == 12:
            current_date = datetime(current_date.year + 1, 1, 1)
        else:
            current_date = datetime(current_date.year, current_date.month + 1, 1)

    return first_of_month_dates

dates_str_lst = generate_first_of_month_dates(start_date_str, end_date_str)
print(dates_str_lst)

['2023-01-01', '2023-02-01', '2023-03-01', '2023-04-01', '2023-05-01', '2023-06-01', '2023-07-01', '2023-08-01', '2023-09-01', '2023-10-01', '2023-11-01', '2023-12-01', '2024-01-01', '2024-02-01', '2024-03-01', '2024-04-01', '2024-05-01', '2024-06-01', '2024-07-01', '2024-08-01', '2024-09-01', '2024-10-01', '2024-11-01', '2024-12-01', '2025-01-01', '2025-02-01', '2025-03-01', '2025-04-01', '2025-05-01', '2025-06-01', '2025-07-01', '2025-08-01', '2025-09-01', '2025-10-01', '2025-11-01']


In [ ]:
# BRONZE
# create bronze datalake - lms
bronze_lms_directory = "datamart/bronze/lms/"

if not os.path.exists(bronze_lms_directory):
    os.makedirs(bronze_lms_directory)

# run bronze backfill - lms
for date_str in dates_str_lst:
    utils.data_processing_bronze_table.process_bronze_lms_table(date_str, bronze_lms_directory, spark)



In [ ]:
# bronze - financials and attributes
table_dict = {
    'source': ['data/features_attributes.csv', 'data/features_financials.csv'],
    'directory': ['datamart/bronze/attributes/', 'datamart/bronze/financials/'],
    'filename': ['cust_attr', 'cust_fin']
             }

for i in range(len(table_dict['source'])):
    if not os.path.exists(table_dict['directory'][i]):
        os.makedirs(table_dict['directory'][i])
    utils.data_processing_bronze_table.process_bronze_other_tables(
        table_dict['source'][i], 
        table_dict['directory'][i], 
        table_dict['filename'][i], 
        spark
    )

In [ ]:
import importlib
import utils.data_processing_bronze_table

importlib.reload(utils.data_processing_bronze_table)

# bronze - clickstream
bronze_clickstream_directory = "datamart/bronze/clickstream/"

if not os.path.exists(bronze_clickstream_directory):
    os.makedirs(bronze_clickstream_directory)

start_date_str = "2023-01-01"
end_date_str = "2024-12-01"
dates_str_lst = generate_first_of_month_dates(start_date_str, end_date_str)
print(dates_str_lst)

for date_str in dates_str_lst:
    utils.data_processing_bronze_table.process_bronze_clickstream_table(
        date_str, 
        bronze_clickstream_directory, 
        spark
    )

# SILVER (working)

## EDA

In [ ]:
attr_df = spark.read.csv('data/features_attributes.csv', header = True, inferSchema = True)
fin_df = spark.read.csv('data/features_financials.csv', header = True, inferSchema = True)

In [ ]:
from pyspark.sql.functions import col, sum, isnan, when
def spark_info(df):
    print(f"Rows: {df.count()}, Columns: {len(df.columns)}")
    print(f"\nSchema:")
    df.printSchema()
    print(f"Null counts:")
    df.select([
        sum(when(col(c).isNull(), 1).otherwise(0)).alias(c)
        for c in df.columns
    ]).show()


### Features_attributes

In [ ]:
spark_info(attr_df)

print(f'\nDescribe table:')
print(attr_df.describe().show())

In [ ]:
from pyspark.sql.functions import countDistinct

attr_df.select(
    countDistinct("Customer_ID").alias("custID_uinque"),
    countDistinct("Occupation").alias("occu_unique"),
    countDistinct("SSN").alias("ssn_unique")
).show()

In [ ]:
attr_df.groupBy("Occupation").count().orderBy("count", ascending = False).show()

In [ ]:
attr_df.groupBy("SSN").count().orderBy("count", ascending = False).show(5)

In [ ]:
# Convert just the one column to pandas
age_pd = attr_df.select("Age").toPandas()

# convert datatype
age_pd["Age"] = age_pd["Age"].str.strip().str.replace("_", "").astype(int)


# Plot histogram
plt.figure(figsize=(10, 6))
plt.hist(age_pd["Age"], bins=40, edgecolor="black")
plt.title("Distribution of Age")
plt.xlabel("Age")
plt.ylabel("Count")
plt.tight_layout()
plt.show()

In [ ]:
age_pd[age_pd['Age'] > 100].sort_values('Age')

In [ ]:
attr_df[attr_df['Occupation'] == '_______'].show()

In [43]:
def process_silver_attr(bronze_directory, silver_directory, spark):
    # connect to bronze table
    df = spark.read.csv(bronze_directory, header = True, inferSchema = True)


    # remove trailing "_" in age
    df = df.withColumn("Age", F.regexp_replace(col("Age"), "_$", ""))

    # enforce schema
    column_type_map = {
        "Customer_ID": StringType(),
        "Name": StringType(), 
        "Age": IntegerType(), 
        "SSN": StringType(), 
        "Occupation": StringType(),
        "snapshot_date": DateType()
    }

    for column, new_type in column_type_map.items():
        df = df.withColumn(column, col(column).cast(new_type))

    # replace invalid SSN with null
    df = df.withColumn(
        "SSN",
        F.when(col("SSN").rlike(r"^\d{3}-\d{2}-\d{4}$"), col("SSN")).otherwise(None)
    )

    # replace invalid age with null
    df = df.withColumn(
        "Age",
        F.when((col("Age") > 0) & (col("Age") <= 120), col("Age")).otherwise(None)
    ) 

    # replace invalid occupation with null
    df = df.withColumn(
        "Occupation",
        F.when(col("Occupation") == '_______', None).otherwise(col("Occupation"))
    )

    print("feature_attributes: silver layer cleaning done")

    # create directory path
    if not os.path.exists(silver_directory):
        os.makedirs(silver_directory)
    
    # save to datamart
    filepath = silver_directory + 'silver_cust_attr' + '.parquet'
    df.write.mode('overwrite').parquet(filepath)
    print("saved to: ", filepath)

    return df



In [44]:
process_silver_attr('datamart/bronze/attributes/bronze_cust_attr.csv', 
                    'datamart/silver/attributes/', 
                    spark)

feature_attributes: silvery layer cleaning done


saved to:  datamart/silver/attributes/silver_cust_attr.parquet


DataFrame[Customer_ID: string, Name: string, Age: int, SSN: string, Occupation: string, snapshot_date: date]

In [ ]:
temp = process_silver_attr('data/features_attributes.csv', 'datamart/silver/cust_attr/', spark)

In [ ]:
spark_info(temp)

print(f'\nDescribe table:')
print(temp.describe().show())

In [ ]:
# temp = process_silver_attr(
#     'datamart/bronze/attributes/bronze_cust_attr.csv',
#     'datamart/silver/silver_cust_attr.csv', 
#     spark)

### Features_financials

In [ ]:
spark_info(fin_df)

print(f'\nDescribe table:')
print(fin_df.describe().show())

In [46]:
def count_loan_type(entry, loan_type):
    if entry is None:
        return 0
    cleaned = re.sub(r",\s+and\s+", ", ", entry)
    loans = [loan.strip() for loan in cleaned.split(", ")]
    return loans.count(loan_type)

def process_silver_fin(bronze_directory, silver_directory, spark):
    # connect to bronze table
    df = spark.read.csv(bronze_directory, header = True, inferSchema = True)
    
    
    # remove trailing "_" in specified columns
    cols_to_clean = ["Annual_Income", "Num_of_Loan", "Num_of_Delayed_Payment", 
                    "Changed_Credit_Limit", "Outstanding_Debt", "Amount_invested_monthly",
                    "Monthly_Balance"]
    
    df = df.select(
        [
            F.regexp_replace(col(c), r"^_+|_+$", "").alias(c)
            if c in cols_to_clean
            else col(c)
            for c in df.columns
        ]
    )
    
    
    # convert Credit_History_Age to integer (years)
    df = df.withColumn("years", F.regexp_extract(col("Credit_History_Age"), r"(\d+)\s+Year", 1).cast("integer")) \
           .withColumn("months", F.regexp_extract(col("Credit_History_Age"), r"(\d+)\s+Month", 1).cast("integer")) \
           .withColumn("Credit_History_Age_Years", F.round(col("years") + col("months") / 12, 2)) \
           .drop("years", "months", "Credit_History_Age")
    
    
    # enforce schema
    column_type_map = {
        'Customer_ID': StringType(),
        'Annual_Income': FloatType(), 
        'Monthly_Inhand_Salary': FloatType(), 
        'Num_Bank_Accounts': IntegerType(), 
        'Num_Credit_Card': IntegerType(),
        'Interest_Rate': FloatType(),
        'Num_of_Loan': IntegerType(), 
        'Type_of_Loan': StringType(), 
        'Delay_from_due_date': IntegerType(), 
        'Num_of_Delayed_Payment': IntegerType(), 
        'Changed_Credit_Limit': FloatType(), 
        'Num_Credit_Inquiries': IntegerType(), 
        'Credit_Mix': StringType(), 
        'Outstanding_Debt': FloatType(),
        'Credit_Utilization_Ratio': FloatType(),
        'Credit_History_Age_Years': FloatType(), 
        'Payment_of_Min_Amount': StringType(), 
        'Total_EMI_per_month': FloatType(),
        'Amount_invested_monthly': FloatType(),
        'Payment_Behaviour': StringType(),
        'Monthly_Balance': FloatType(),
        'snapshot_date': DateType()
    }
    
    for column, new_type in column_type_map.items():
        df = df.withColumn(column, col(column).cast(new_type))
    
    
    # handle anomalous values for specified quantitative columns
    cols_iqr = ['Num_Bank_Accounts', 'Num_Credit_Card', 'Num_of_Loan', 'Delay_from_due_date', 
                'Num_of_Delayed_Payment', 'Num_Credit_Inquiries']
    
    for c in cols_iqr:
        df = df.withColumn(c, F.when(col(c)<0, None).otherwise(col(c)))
        q1 = df.approxQuantile(c, [0.25], 0.0)[0]
        q3 = df.approxQuantile(c, [0.75], 0.0)[0]
        iqr = q3 - q1
        upper = q3 + 1.5 * iqr
        df = df.withColumn(
            c,
            F.when((col(c) >= 0) & (col(c) <= upper), col(c)).otherwise(None)
        )

    # handle negative values for other quantitative columns
    cols_others = ['Annual_Income', 'Interest_Rate', 'Outstanding_Debt', 'Credit_Utilization_Ratio',
                   'Total_EMI_per_month', 'Amount_invested_monthly', 'Monthly_Balance']

    for c in cols_others:
        df = df.withColumn(c, F.when(col(c)<0, None).otherwise(col(c)))


    # replace Type_of_Loan with columns specifying type of loan and the count
    ## 1. get unique loan types 
    loan_series = df.select("Type_of_Loan").dropna().toPandas()["Type_of_Loan"]
    
    all_loans = []
    for entry in loan_series:
        cleaned = re.sub(r",\s+and\s+", ", ", entry)
        loans = [loan.strip() for loan in cleaned.split(", ")]
        all_loans.extend(loans)
    
    unique_loans = sorted(set(all_loans))
    
    ## 2. add a column for each loan type
    for loan_type in unique_loans:
        col_name = "Loan_" + loan_type.replace(" ", "_").replace("-", "_")
        
        count_udf = udf(lambda x: count_loan_type(x, loan_type), IntegerType())
        
        df = df.withColumn(col_name, count_udf(col("Type_of_Loan")))
    
    ## 3. drop original column
    df = df.drop("Type_of_Loan")
    
    
    # replace "_" in Credit_Mix 
    df = df.withColumn("Credit_Mix",
        F.when(col("Credit_Mix") == "_", None).otherwise(col("Credit_Mix"))
    )
    
    # separate Payment_Behaviour into 2 columns
    valid_pattern = r"^[A-Za-z]+_spent_[A-Za-z]+_value_payments$"
    
    df = (
        df
        .withColumn("parts", F.split(col("Payment_Behaviour"), "_"))
        .withColumn("Spending_Behaviour",
                    F.when(col("Payment_Behaviour").rlike(valid_pattern), col("parts")[0]).otherwise(None)
                   )
        .withColumn("Payments_Size",
                    F.when(col("Payment_Behaviour").rlike(valid_pattern), col("parts")[2]).otherwise(None)
                   )
        .drop("Payment_Behaviour", "parts")
    )

    print("feature_financials: silver layer cleaning done")
    
    # create directory path
    if not os.path.exists(silver_directory):
        os.makedirs(silver_directory)
    
    # save to datamart
    filepath = silver_directory + 'silver_cust_fin' + '.parquet'
    df.write.mode('overwrite').parquet(filepath)
    print("saved to: ", filepath)
    
    return df




In [47]:
process_silver_fin('datamart/bronze/financials/bronze_cust_fin.csv', 
                   'datamart/silver/financials/', 
                    spark)

feature_financials: silver layer cleaning done


saved to:  datamart/silver/financials/silver_cust_fin.parquet


DataFrame[Customer_ID: string, Annual_Income: float, Monthly_Inhand_Salary: float, Num_Bank_Accounts: int, Num_Credit_Card: int, Interest_Rate: float, Num_of_Loan: int, Delay_from_due_date: int, Num_of_Delayed_Payment: int, Changed_Credit_Limit: float, Num_Credit_Inquiries: int, Credit_Mix: string, Outstanding_Debt: float, Credit_Utilization_Ratio: float, Payment_of_Min_Amount: string, Total_EMI_per_month: float, Amount_invested_monthly: float, Monthly_Balance: float, snapshot_date: date, Credit_History_Age_Years: float, Loan_Auto_Loan: int, Loan_Credit_Builder_Loan: int, Loan_Debt_Consolidation_Loan: int, Loan_Home_Equity_Loan: int, Loan_Mortgage_Loan: int, Loan_Not_Specified: int, Loan_Payday_Loan: int, Loan_Personal_Loan: int, Loan_Student_Loan: int, Spending_Behaviour: string, Payments_Size: string]

In [ ]:
def count_loan_type(entry, loan_type):
    if entry is None:
        return 0
    cleaned = re.sub(r",\s+and\s+", ", ", entry)
    loans = [loan.strip() for loan in cleaned.split(", ")]
    return loans.count(loan_type)


    
# connect to bronze table
df = spark.read.csv('datamart/bronze/financials/bronze_cust_fin.csv', header = True, inferSchema = True)


# remove trailing "_" in specified columns
cols_to_clean = ["Annual_Income", "Num_of_Loan", "Num_of_Delayed_Payment", 
                "Changed_Credit_Limit", "Outstanding_Debt"]

df = df.select(
    [F.regexp_replace(col(c), "_$", "").alias(c) if c in cols_to_clean 
     else col(c) 
     for c in df.columns]
)


# convert Credit_History_Age to integer (years)
df = df.withColumn("years", F.regexp_extract(col("Credit_History_Age"), r"(\d+)\s+Year", 1).cast("integer")) \
       .withColumn("months", F.regexp_extract(col("Credit_History_Age"), r"(\d+)\s+Month", 1).cast("integer")) \
       .withColumn("Credit_History_Age_Years", F.round(col("years") + col("months") / 12, 2)) \
       .drop("years", "months", "Credit_History_Age")


# enforce schema
column_type_map = {
    'Customer_ID': StringType(),
    'Annual_Income': FloatType(), 
    'Monthly_Inhand_Salary': FloatType(), 
    'Num_Bank_Accounts': IntegerType(), 
    'Num_Credit_Card': IntegerType(),
    'Interest_Rate': FloatType(),
    'Num_of_Loan': IntegerType(), 
    'Type_of_Loan': StringType(), 
    'Delay_from_due_date': IntegerType(), 
    'Num_of_Delayed_Payment': IntegerType(), 
    'Changed_Credit_Limit': FloatType(), 
    'Num_Credit_Inquiries': IntegerType(), 
    'Credit_Mix': StringType(), 
    'Outstanding_Debt': FloatType(),
    'Credit_Utilization_Ratio': FloatType(),
    'Credit_History_Age_Years': FloatType(), 
    'Payment_of_Min_Amount': StringType(), 
    'Total_EMI_per_month': FloatType(),
    'Amount_invested_monthly': FloatType(),
    'Payment_Behaviour': StringType(),
    'Monthly_Balance': FloatType(),
    'snapshot_date': DateType()
}

for column, new_type in column_type_map.items():
    df = df.withColumn(column, col(column).cast(new_type))


# handle anomalous values for specified quantitative columns
cols_iqr = ['Num_Bank_Accounts', 'Num_Credit_Card', 'Num_of_Loan', 'Delay_from_due_date', 
            'Num_of_Delayed_Payment', 'Num_Credit_Inquiries']

for c in cols_iqr:
    df = df.withColumn(c, F.when(col(c)<0, None).otherwise(col(c)))
    q1 = df.approxQuantile(c, [0.25], 0.0)[0]
    q3 = df.approxQuantile(c, [0.75], 0.0)[0]
    iqr = q3 - q1
    upper = q3 + 1.5 * iqr
    df = df.withColumn(
        c,
        when((col(c) >= 0) & (col(c) <= upper), col(c)).otherwise(None)
    )


# replace Type_of_Loan with columns specifying type of loan and the count
## 1. get unique loan types 
loan_series = df.select("Type_of_Loan").dropna().toPandas()["Type_of_Loan"]

all_loans = []
for entry in loan_series:
    cleaned = re.sub(r",\s+and\s+", ", ", entry)
    loans = [loan.strip() for loan in cleaned.split(", ")]
    all_loans.extend(loans)

unique_loans = sorted(set(all_loans))

## 2. add a column for each loan type
for loan_type in unique_loans:
    col_name = "Loan_" + loan_type.replace(" ", "_").replace("-", "_")
    
    count_udf = udf(lambda x: count_loan_type(x, loan_type), IntegerType())
    
    df = df.withColumn(col_name, count_udf(col("Type_of_Loan")))

## 3. drop original column
df = df.drop("Type_of_Loan")


# replace "_" in Credit_Mix 
df = df.withColumn("Credit_Mix",
    F.when(col("Credit_Mix") == "_", None).otherwise(col("Credit_Mix"))
)

# separate Payment_Behaviour into 2 columns
valid_pattern = r"^[A-Za-z]+_spent_[A-Za-z]+_value_payments$"

df = (
    df
    .withColumn("parts", F.split(col("Payment_Behaviour"), "_"))
    .withColumn("Spending_Behaviour",
                F.when(col("Payment_Behaviour").rlike(valid_pattern), col("parts")[0]).otherwise(None)
               )
    .withColumn("Payments_Size",
                F.when(col("Payment_Behaviour").rlike(valid_pattern), col("parts")[2]).otherwise(None)
               )
    .drop("Payment_Behaviour", "parts")
)

In [ ]:

df.toPandas().describe(include = 'object')

In [ ]:
df_pd['Payment_Behaviour'].value_counts()

In [ ]:
import re


# Step 1 — get unique loan types (use pandas for this)
loan_series = df.select("Type_of_Loan").dropna().toPandas()["Type_of_Loan"]

all_loans = []
for entry in loan_series:
    cleaned = re.sub(r",\s+and\s+", ", ", entry)
    loans = [loan.strip() for loan in cleaned.split(", ")]
    all_loans.extend(loans)

unique_loans = sorted(set(all_loans))
print(unique_loans)



In [ ]:
df_pd = df.toPandas()
df_pd.describe(include = 'object')

In [ ]:
import re

# Collect all loan values into a list
loan_series = df_pd["Type_of_Loan"].dropna()

# Split each row and clean up
all_loans = []
for entry in loan_series:
    # Remove " and " before last item, then split by ", "
    cleaned = re.sub(r",\s+and\s+", ", ", entry)
    loans = [loan.strip() for loan in cleaned.split(", ")]
    all_loans.extend(loans)

# Get unique values
unique_loans = set(all_loans)
print(f"Unique loan types ({len(unique_loans)}):")
for loan in sorted(unique_loans):
    print(f"  - {loan}")

In [ ]:
temp = df_pd.copy()
temp["Total_EMI_per_month"] = temp["Total_EMI_per_month"].where(temp["Total_EMI_per_month"] >= 0, None)

# Calculate IQR
Q1 = temp["Total_EMI_per_month"].quantile(0.25)
Q3 = temp["Total_EMI_per_month"].quantile(0.75)
IQR = Q3 - Q1
lower = Q1 - 1.5 * IQR
upper = Q3 + 1.5 * IQR

print(f"Q1: {Q1}")
print(f"Q3: {Q3}")
print(f"IQR: {IQR}")
print(f"Lower: {lower}")
print(f"Upper: {upper}")

# Delay_from_due_date	Num_of_Delayed_Payment	

In [ ]:
df_pd[df_pd["Num_Credit_Inquiries"]>19].sort_values("Num_Credit_Inquiries")

In [ ]:
# Convert to pandas
credit_card_pd = df.select("Total_EMI_per_month").toPandas()

# Sort values and calculate cumulative percentage
sorted_vals = np.sort(credit_card_pd["Total_EMI_per_month"])
cumulative = np.arange(1, len(sorted_vals) + 1) / len(sorted_vals) * 100

# Plot
plt.figure(figsize=(10, 6))
plt.plot(sorted_vals, cumulative, marker=".", linestyle="-")
plt.title("Cumulative Distribution of Num_of_Loan")
plt.xlabel("Num_of_Loan")
plt.ylabel("Cumulative %")
plt.grid(True)
plt.tight_layout()
plt.show()

In [ ]:
df_pd["Total_EMI_per_month"].describe()
df_pd["Total_EMI_per_month"].value_counts().sort_index()

# Plot to see distribution
df_pd["Total_EMI_per_month"].hist(bins=30)

In [ ]:
df_pd["Delay_from_due_date"].describe()
df_pd["Delay_from_due_date"].value_counts().sort_index()

# Plot to see distribution
df_pd["Delay_from_due_date"].hist(bins=30)

In [ ]:
pd.set_option('display.float_format', '{:.2f}'.format)
df_pd = df.toPandas()
df_pd.describe()

In [ ]:
df_pd.describe(include = 'object')

### Check attr and fin????

In [40]:
process_silver_attr('datamart/bronze/attributes/bronze_cust_attr.csv', 
                                  'datamart/silver/attributes/', 
                                  spark)

# attr_silver.toPandas().info()

DataFrame[Customer_ID: string, Name: string, Age: int, SSN: string, Occupation: string, snapshot_date: date]

In [35]:
attr_pd = attr_silver.toPandas()

In [28]:
fin_silver = process_silver_fin('datamart/bronze/financials/bronze_cust_fin.csv',
                                'datamart/silver/financials/',
                                spark)

fin_silver.toPandas().info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 12500 entries, 0 to 12499
Data columns (total 31 columns):
 #   Column                        Non-Null Count  Dtype  
---  ------                        --------------  -----  
 0   Customer_ID                   12500 non-null  object 
 1   Annual_Income                 12500 non-null  float32
 2   Monthly_Inhand_Salary         12500 non-null  float32
 3   Num_Bank_Accounts             12329 non-null  float64
 4   Num_Credit_Card               12204 non-null  float64
 5   Interest_Rate                 12500 non-null  float32
 6   Num_of_Loan                   11933 non-null  float64
 7   Delay_from_due_date           11908 non-null  float64
 8   Num_of_Delayed_Payment        12311 non-null  float64
 9   Changed_Credit_Limit          12246 non-null  float32
 10  Num_Credit_Inquiries          12305 non-null  float64
 11  Credit_Mix                    9889 non-null   object 
 12  Outstanding_Debt              12500 non-null  float32
 13  C

In [36]:
fin_pd = fin_silver.toPandas()

In [38]:
# Get sets of (customer_id, snapshot_date) from each df
attr_keys = set(zip(attr_pd["Customer_ID"], attr_pd["snapshot_date"]))
fin_keys = set(zip(fin_pd["Customer_ID"], fin_pd["snapshot_date"]))

# Find mismatches
in_attr_not_fin = attr_keys - fin_keys
in_fin_not_attr = fin_keys - attr_keys

print(f"In attributes but NOT in financials: {len(in_attr_not_fin)}")
print(f"In financials but NOT in attributes: {len(in_fin_not_attr)}")

In attributes but NOT in financials: 0
In financials but NOT in attributes: 0


### LMS

In [48]:
lms_df = spark.read.csv('data/lms_loan_daily.csv', header = True, inferSchema = True)

In [49]:
df_pd = lms_df.toPandas()

In [50]:
df_pd.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 137500 entries, 0 to 137499
Data columns (total 11 columns):
 #   Column           Non-Null Count   Dtype  
---  ------           --------------   -----  
 0   loan_id          137500 non-null  object 
 1   Customer_ID      137500 non-null  object 
 2   loan_start_date  137500 non-null  object 
 3   tenure           137500 non-null  int32  
 4   installment_num  137500 non-null  int32  
 5   loan_amt         137500 non-null  int32  
 6   due_amt          137500 non-null  float64
 7   paid_amt         137500 non-null  float64
 8   overdue_amt      137500 non-null  float64
 9   balance          137500 non-null  float64
 10  snapshot_date    137500 non-null  object 
dtypes: float64(4), int32(3), object(4)
memory usage: 10.0+ MB


In [51]:
df_pd.describe()

,tenure,installment_num,loan_amt,due_amt,paid_amt,overdue_amt,balance
count,137500.00,137500.00,137500.00,137500.00,137500.00,137500.00,137500.00
mean,10.00,5.00,10000.00,909.09,711.81,871.91,5871.91
std,0.00,3.16,0.00,287.48,485.73,2002.67,3070.78
min,10.00,0.00,10000.00,0.00,0.00,0.00,0.00
25%,10.00,2.00,10000.00,1000.00,0.00,0.00,3000.00
50%,10.00,5.00,10000.00,1000.00,1000.00,0.00,7000.00
75%,10.00,8.00,10000.00,1000.00,1000.00,0.00,8000.00
max,10.00,10.00,10000.00,1000.00,4000.00,10000.00,10000.00


In [53]:
df_pd.describe(include = 'object')

,loan_id,Customer_ID,loan_start_date,snapshot_date
count,137500,137500,137500,137500
unique,12500,12500,25,35
top,CUS_0xffd_2024_03_01,CUS_0xffd,2024-08-01,2025-01-01
freq,11,11,5973,5539


In [56]:
df_pd.groupby('Customer_ID')['loan_id'].nunique().reset_index() \
     .rename(columns={'loan_id': 'loan_count'}) \
     .sort_values('loan_count', ascending=False)

,Customer_ID,loan_count
12499,CUS_0xffd,1
0,CUS_0x1000,1
1,CUS_0x1009,1
2,CUS_0x100b,1
12483,CUS_0xfbd,1
...,...,...
8,CUS_0x102d,1
7,CUS_0x1026,1
6,CUS_0x1018,1
5,CUS_0x1015,1


In [ ]:
# loan_prod: tenure, loan_amt, due_amt (ie monthly repayment)
# loan_dim: loan_id, loan_start_date
# loan_daily: loan_id, customer_ID, snapshot_date, installment_num, paid_amt, overdue_amt, balance


In [57]:
def process_silver_lms(snapshot_date_str, bronze_lms_directory, silver_loan_daily_directory, spark):
    # prepare arguments
    snapshot_date = datetime.strptime(snapshot_date_str, "%Y-%m-%d")
    
    # connect to bronze table
    partition_name = "bronze_loan_daily_" + snapshot_date_str.replace('-','_') + '.csv'
    filepath = bronze_lms_directory + partition_name
    df = spark.read.csv(filepath, header=True, inferSchema=True)
    print('loaded from:', filepath, 'row count:', df.count())
    
    # clean data: enforce schema / data type
    column_type_map = {
        "loan_id": StringType(),
        "Customer_ID": StringType(),
        "loan_start_date": DateType(),
        "tenure": IntegerType(),
        "installment_num": IntegerType(),
        "loan_amt": FloatType(),
        "due_amt": FloatType(),
        "paid_amt": FloatType(),
        "overdue_amt": FloatType(),
        "balance": FloatType(),
        "snapshot_date": DateType()
    }
    
    for column, new_type in column_type_map.items():
        df = df.withColumn(column, col(column).cast(new_type))
    
    # augment data: add month on book
    df = df.withColumn("mob", col("installment_num").cast(IntegerType()))
    
    # augment data: add days past due
    df = df.withColumn("installments_missed", F.ceil(col("overdue_amt") / col("due_amt")).cast(IntegerType())).fillna(0)
    df = df.withColumn("first_missed_date", F.when(col("installments_missed") > 0, F.add_months(col("snapshot_date"), -1 * col("installments_missed"))).cast(DateType()))
    df = df.withColumn("dpd", F.when(col("overdue_amt") > 0.0, F.datediff(col("snapshot_date"), col("first_missed_date"))).otherwise(0).cast(IntegerType()))
    
    # keep full df for loan_prod and loan_dim
    df_full = df
    
    # drop columns extracted to loan_prod and loan_dim
    df_dropped = df.drop("loan_start_date", "tenure", "loan_amt", "due_amt")
    
    # save silver loan daily table (dropped version)
    partition_name = "silver_loan_daily_" + snapshot_date_str.replace('-','_') + '.parquet'
    filepath = silver_loan_daily_directory + partition_name
    df_dropped.write.mode("overwrite").parquet(filepath)
    print('saved to:', filepath)
    
    # return both — full for loan_prod/loan_dim, dropped for reference
    return df_full, df_dropped


def process_silver_loan_prod(dfs, silver_directory):
    # union all monthly dfs and get distinct
    df_all = dfs[0]
    for df in dfs[1:]:
        df_all = df_all.union(df)
    
    loan_prod = df_all.select("tenure", "loan_amt", "due_amt").distinct().filter(col("due_amt") > 0)
    
    filepath = silver_directory + "loan_prod.parquet"
    loan_prod.write.mode("overwrite").parquet(filepath)
    print(f"Saved loan_prod (tenure, principal, monthly repayment): {loan_prod.count()} row(s) to {filepath}")
    
    return loan_prod


def process_silver_loan_dim(dfs, silver_directory):
    # union all monthly dfs and get distinct loan_id + loan_start_date
    df_all = dfs[0]
    for df in dfs[1:]:
        df_all = df_all.union(df)
    
    loan_dim = df_all.select("loan_id", "loan_start_date").distinct()
    
    filepath = silver_directory + "loan_dim.parquet"
    loan_dim.write.mode("overwrite").parquet(filepath)
    print(f"Saved loan_dim (loan id and start date): {loan_dim.count()} row(s) to {filepath}")
    
    return loan_dim

In [59]:
# process monthly silver tables
dfs_full = []
for date_str in dates_str_lst:
    
    df_full, df_dropped = process_silver_lms(
        date_str, 'datamart/bronze/lms/', 'datamart/silver/loan_daily/', spark
    )
    
    dfs_full.append(df_full)

# get loan_prod and loan_dim tables
process_silver_loan_prod(dfs_full, 'datamart/silver/loan_daily/')
process_silver_loan_dim(dfs_full, 'datamart/silver/loan_daily/')

loaded from: datamart/bronze/lms/bronze_loan_daily_2023_01_01.csv row count: 530
saved to: datamart/silver/loan_daily/silver_loan_daily_2023_01_01.parquet
loaded from: datamart/bronze/lms/bronze_loan_daily_2023_02_01.csv row count: 1031
saved to: datamart/silver/loan_daily/silver_loan_daily_2023_02_01.parquet
loaded from: datamart/bronze/lms/bronze_loan_daily_2023_03_01.csv row count: 1537
saved to: datamart/silver/loan_daily/silver_loan_daily_2023_03_01.parquet
loaded from: datamart/bronze/lms/bronze_loan_daily_2023_04_01.csv row count: 2047
saved to: datamart/silver/loan_daily/silver_loan_daily_2023_04_01.parquet
loaded from: datamart/bronze/lms/bronze_loan_daily_2023_05_01.csv row count: 2568
saved to: datamart/silver/loan_daily/silver_loan_daily_2023_05_01.parquet
loaded from: datamart/bronze/lms/bronze_loan_daily_2023_06_01.csv row count: 3085
saved to: datamart/silver/loan_daily/silver_loan_daily_2023_06_01.parquet
loaded from: datamart/bronze/lms/bronze_loan_daily_2023_07_01.csv

Saved loan_prod (tenure, principal, monthly repayment): 1 row(s) to datamart/silver/loan_daily/loan_prod.parquet


Saved loan_dim (loan id and start date): 12500 row(s) to datamart/silver/loan_daily/loan_dim.parquet


DataFrame[loan_id: string, loan_start_date: date]

In [ ]:
# # TO ADD TO MAIN.PY
# # process monthly silver tables
# dfs_full = []
# for date_str in dates_str_lst:
    
#     df_full, df_dropped = utils.data_processing_silver_table.process_silver_lms(
#         date_str, bronze_lms_directory, silver_loan_daily_directory, spark
#     )
    
#     dfs_full.append(df_full)

# # get loan_prod and loan_dim tables
# utils.data_processing_silver_table.process_silver_loan_prod(dfs_full, silver_directory)
# utils.data_processing_silver_table.process_silver_loan_dim(dfs_full, silver_directory)

In [63]:
# # loan_dim
# loan_dim = spark.read.parquet("datamart/silver/loan_daily/loan_dim.parquet")
# loan_dim.printSchema()
# display(loan_dim.limit(5).toPandas())
# display(loan_dim.summary().toPandas())

# # loan_prod
# loan_prod = spark.read.parquet("datamart/silver/loan_daily/loan_prod.parquet")
# loan_prod.printSchema()
# display(loan_prod.limit(5).toPandas())
# display(loan_prod.summary().toPandas())

# # silver_loan_daily
# silver_loan_daily = spark.read.parquet("datamart/silver/loan_daily/silver_loan_daily_2024_04_01.parquet")
# silver_loan_daily.printSchema()
# display(silver_loan_daily.limit(20).toPandas())
# display(silver_loan_daily.summary().toPandas())

### Clickstream

In [64]:
clicks_df = spark.read.csv('data/feature_clickstream.csv', header = True, inferSchema = True)

In [66]:
clicks_pd = clicks_df.toPandas()

In [69]:
clicks_pd.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 215376 entries, 0 to 215375
Data columns (total 22 columns):
 #   Column         Non-Null Count   Dtype 
---  ------         --------------   ----- 
 0   fe_1           215376 non-null  int32 
 1   fe_2           215376 non-null  int32 
 2   fe_3           215376 non-null  int32 
 3   fe_4           215376 non-null  int32 
 4   fe_5           215376 non-null  int32 
 5   fe_6           215376 non-null  int32 
 6   fe_7           215376 non-null  int32 
 7   fe_8           215376 non-null  int32 
 8   fe_9           215376 non-null  int32 
 9   fe_10          215376 non-null  int32 
 10  fe_11          215376 non-null  int32 
 11  fe_12          215376 non-null  int32 
 12  fe_13          215376 non-null  int32 
 13  fe_14          215376 non-null  int32 
 14  fe_15          215376 non-null  int32 
 15  fe_16          215376 non-null  int32 
 16  fe_17          215376 non-null  int32 
 17  fe_18          215376 non-null  int32 
 18  fe_1

In [70]:
clicks_pd.describe()

,fe_1,fe_2,fe_3,fe_4,fe_5,fe_6,fe_7,fe_8,fe_9,fe_10,fe_11,fe_12,fe_13,fe_14,fe_15,fe_16,fe_17,fe_18,fe_19,fe_20
count,215376.00,215376.00,215376.00,215376.00,215376.00,215376.00,215376.00,215376.00,215376.00,215376.00,215376.00,215376.00,215376.00,215376.00,215376.00,215376.00,215376.00,215376.00,215376.00,215376.00
mean,101.41,103.10,104.33,105.65,107.00,103.24,107.07,110.72,114.41,117.78,99.62,99.82,100.42,100.03,99.60,99.86,100.13,100.13,100.34,100.05
std,99.83,99.93,100.60,100.33,100.69,100.27,100.32,100.24,100.19,100.81,100.34,100.59,100.88,101.07,101.15,100.75,101.30,102.23,102.67,103.59
min,-378.00,-356.00,-399.00,-307.00,-343.00,-321.00,-368.00,-361.00,-328.00,-317.00,-375.00,-344.00,-355.00,-394.00,-351.00,-342.00,-329.00,-344.00,-401.00,-354.00
25%,34.00,36.00,36.00,38.00,39.00,36.00,39.00,43.00,47.00,50.00,32.00,32.00,33.00,32.00,31.00,32.00,32.00,31.00,31.00,30.00
50%,102.00,103.00,104.00,106.00,107.00,103.00,107.00,111.00,115.00,118.00,100.00,100.00,101.00,100.00,100.00,100.00,100.00,100.00,100.00,100.00
75%,169.00,171.00,172.00,173.00,175.00,171.00,174.00,179.00,182.00,186.00,167.00,168.00,168.00,168.00,168.00,168.00,169.00,169.00,170.00,170.00
max,541.00,560.00,583.00,562.00,570.00,565.00,537.00,573.00,577.00,537.00,613.00,550.00,530.00,583.00,597.00,554.00,516.00,551.00,560.00,547.00


In [83]:
def process_silver_clickstream(snapshot_date_str, bronze_directory, silver_directory, spark):
    # connect to bronze table
    partition_name = "bronze_clickstream_daily_" + snapshot_date_str.replace('-','_') + '.csv'
    filepath = bronze_directory + partition_name
    df = spark.read.csv(filepath, header=True, inferSchema=True)
    print('loaded from:', filepath, 'row count:', df.count())

    
    # enforce schema
    for c in df.columns:
        if c.startswith("fe_"):
            df = df.withColumn(c, col(c).cast(IntegerType()))
    
    df = df.withColumn("Customer_ID", col("Customer_ID").cast(StringType()))
    df = df.withColumn("snapshot_date", col("snapshot_date").cast(DateType()))

    
    # check and drop duplicates
    df = df.dropDuplicates()


    # save silver table
    partition_name = "silver_clickstream_daily_" + snapshot_date_str.replace('-','_') + '.parquet'
    filepath = silver_directory + partition_name
    df.write.mode("overwrite").parquet(filepath)
    # df.toPandas().to_parquet(filepath,
    #           compression='gzip')
    print('saved to:', filepath)
    
    return df


In [85]:
start_date_str = '2023-01-01'
end_date_str = '2024-12-01'

dates_str_lst = generate_first_of_month_dates(start_date_str, end_date_str)
print("list of dates for clickstream:", dates_str_lst)

silver_clickstream_directory = 'datamart/silver/clickstream/'
bronze_clickstream_directory = 'datamart/bronze/clickstream/'

if not os.path.exists(silver_clickstream_directory):
    os.makedirs(silver_clickstream_directory)

for date_str in dates_str_lst:
    process_silver_clickstream(
        date_str, bronze_clickstream_directory, silver_clickstream_directory, spark
    )

list of dates for clickstream: ['2023-01-01', '2023-02-01', '2023-03-01', '2023-04-01', '2023-05-01', '2023-06-01', '2023-07-01', '2023-08-01', '2023-09-01', '2023-10-01', '2023-11-01', '2023-12-01', '2024-01-01', '2024-02-01', '2024-03-01', '2024-04-01', '2024-05-01', '2024-06-01', '2024-07-01', '2024-08-01', '2024-09-01', '2024-10-01', '2024-11-01', '2024-12-01']
loaded from: datamart/bronze/clickstream/bronze_clickstream_daily_2023_01_01.csv row count: 8974
saved to: datamart/silver/clickstream/silver_clickstream_daily_2023_01_01.parquet
loaded from: datamart/bronze/clickstream/bronze_clickstream_daily_2023_02_01.csv row count: 8974
saved to: datamart/silver/clickstream/silver_clickstream_daily_2023_02_01.parquet
loaded from: datamart/bronze/clickstream/bronze_clickstream_daily_2023_03_01.csv row count: 8974
saved to: datamart/silver/clickstream/silver_clickstream_daily_2023_03_01.parquet
loaded from: datamart/bronze/clickstream/bronze_clickstream_daily_2023_04_01.csv row count: 897

In [80]:
# start_date_str = '2023-01-01'
# end_date_str = '2024-12-01'

# dates_str_lst = generate_first_of_month_dates(start_date_str, end_date_str)
# print("list of dates for clickstream:", dates_str_lst)

# silver_clickstream_directory = 'datamart/silver/clickstream/'
# bronze_clickstream_directory = 'datamart/bronze/clickstream/'

# if not os.path.exists(silver_clickstream_directory):
#     os.makedirs(silver_clickstream_directory)

# for date_str in dates_str_lst:
#     utils.data_processing_silver_table.process_silver_clickstream(
#         date_str, bronze_clickstream_directory, spark
#     )


In [ ]:
def process_silver_lms(
    snapshot_date_str, bronze_lms_directory, 
    silver_loan_daily_directory, spark):

# GOLD

In [ ]:
df_attr = spark.read.parquet("datamart/silver/attributes/")
df_fin = spark.read.parquet("datamart/silver/financials/")
df_loan_prod = spark.read.parquet("datamart/silver/loan_daily/loan_prod.parquet")
df_loan_dim = spark.read.parquet("datamart/silver/loan_daily/loan_dim.parquet")

# ── monthly parquet files (read entire folder at once) ─
df_clicks   = spark.read.parquet("datamart/silver/clickstream/")
df_loan     = spark.read.parquet("datamart/silver/loan_daily/")